In [2]:
import pandas as pd
import json
import re
from collections import defaultdict


from google.colab import drive
drive.mount('/content/drive')
df = pd.read_excel("/content/drive/MyDrive/NBEA data collection.xlsx",
                   sheet_name="Main Data Collection Sheet")

Mounted at /content/drive


In [20]:



df.columns = df.columns.str.strip()

df2 = df[[
    "HGVS p.(NP_056493.3)",
    "HGVS c. (NM_015678.4)",
    "mutation type",
    "Splice aa position",
    "IGD exon start",
    "IGD exon end",
    "seizures y/n",
    "Clinical lab classification condensed (VUS /Pathogenic)",
    "Inheritance",
    "age seizure onset"
]].copy()

df2.columns = [
    "protein", "coding", "mut_type", "splice_pos",
    "igd_exon_start", "igd_exon_end",
    "seizures", "path", "inheritance", "onset"
]

# Clean only text columns
for col in ["protein", "coding", "mut_type", "seizures", "path", "inheritance", "onset"]:
    df2[col] = df2[col].fillna("").astype(str)

df2["mut_type"] = df2["mut_type"].str.strip().str.lower()

#  2. Helper functions

def get_aa(p):
    m = re.search(r"(\d+)", str(p))
    return int(m.group(1)) if m else None

def abbr(p):
    p = str(p)
    for a, b in [
        ("Ala","A"),("Arg","R"),("Asn","N"),("Asp","D"),("Cys","C"),
        ("Gln","Q"),("Glu","E"),("Gly","G"),("His","H"),("Ile","I"),
        ("Leu","L"),("Lys","K"),("Met","M"),("Phe","F"),("Pro","P"),
        ("Ser","S"),("Thr","T"),("Trp","W"),("Tyr","Y"),("Val","V"),("Ter","*")
    ]:
        p = p.replace(a, b)
    return p

#  3. Recode variables

df2["aa_pos"] = df2["protein"].apply(get_aa)
df2["splice_pos"] = pd.to_numeric(df2["splice_pos"], errors="coerce")

df2["plot_pos"] = df2.apply(
    lambda r: r["aa_pos"] if pd.notna(r["aa_pos"])
    else (
        int(r["splice_pos"])
        if "splice" in r["mut_type"] and pd.notna(r["splice_pos"])
        else None
    ),
    axis=1
)

df2["mut_type"] = df2["mut_type"].apply(lambda x:
    "missense" if "missense" in x else
    "nonsense" if "nonsense" in x or "stop" in x else
    "frameshift" if "frameshift" in x or "frameshit" in x else
    "splice" if "splice" in x else
    "igd" if "igd" in x else
    x
)

df2["seizure"] = df2["seizures"].str.lower().str.strip().apply(
    lambda x: "Seizure" if x.startswith("y") else
              "No seizure" if x.startswith("n") or x == "no" else
              "Unknown"
)

df2["path_clean"] = df2["path"].str.lower().apply(
    lambda x: "Pathogenic" if "pathogenic" in x else
              "VUS" if "vus" in x else
              "Unknown"
)

df2["inherited"] = df2["inheritance"].str.lower().apply(
    lambda x: True if any(w in x for w in ["maternally", "paternally", "inherited from"]) else False
)

df2["label"] = df2.apply(lambda r:
    abbr(r["protein"]) if r["mut_type"] in ["missense", "nonsense", "frameshift"] and r["protein"]
    else (r["coding"] if "splice" in r["mut_type"] and r["coding"] else ""),
    axis=1
)

pt = df2[
    df2["plot_pos"].notna() &
    df2["mut_type"].isin(["missense", "nonsense", "frameshift", "splice"])
].copy()

pt["plot_pos"] = pt["plot_pos"].astype(int)
pt = pt.reset_index(drop=True)

# 4. Layout constants

W = 1600
PROTEIN_LEN = 2961
LEFT = 80
RIGHT = 1540
PLOT_W = RIGHT - LEFT
BACKBONE_Y = 380

SZ_COLORS = {
    "Seizure": "#D62728",
    "No seizure": "#4878CF",
    "Unknown": "#AAAAAA"
}

PATH_COLORS = {
    "Pathogenic": "#C0392B",
    "VUS": "#F39C12",
    "Unknown": "#AAAAAA"
}

def aa_to_x(aa):
    return LEFT + (aa / PROTEIN_LEN) * PLOT_W

domains = [
    {"name": "D4704", "start": 458,  "end": 937,  "color": "#4C72B0", "wd": False},
    {"name": "WD1",   "start": 1326, "end": 1368, "color": "#2E7D32", "wd": True},
    {"name": "D1088", "start": 1956, "end": 2122, "color": "#F1C40F", "wd": False},
    {"name": "PH",    "start": 2147, "end": 2255, "color": "#8E44AD", "wd": False},
    {"name": "BEACH", "start": 2274, "end": 2563, "color": "#C0392B", "wd": False},
    {"name": "WD2",   "start": 2718, "end": 2761, "color": "#2E7D32", "wd": True},
    {"name": "WD3",   "start": 2778, "end": 2818, "color": "#2E7D32", "wd": True},
    {"name": "WD4",   "start": 2860, "end": 2899, "color": "#2E7D32", "wd": True},
    {"name": "WD5",   "start": 2902, "end": 2941, "color": "#2E7D32", "wd": True},
]

TYPE_Y = {
    "missense":   {"base": BACKBONE_Y - 55,  "dir": -1, "step": 20},
    "nonsense":   {"base": BACKBONE_Y + 55,  "dir":  1, "step": 22},
    "frameshift": {"base": BACKBONE_Y + 105, "dir":  1, "step": 24},
    "splice":     {"base": BACKBONE_Y + 165, "dir":  1, "step": 26},
}
#  5. Compute point positions

groups = defaultdict(list)

for i, row in pt.iterrows():
    side = "top" if row["mut_type"] == "missense" else "bottom"
    groups[(row["plot_pos"], side)].append(i)

points = []

for i, row in pt.iterrows():
    mt = row["mut_type"]
    side = "top" if mt == "missense" else "bottom"
    key = (row["plot_pos"], side)
    idx = groups[key].index(i)
    n = len(groups[key])
    cfg = TYPE_Y[mt]

    y_pt = cfg["base"] + cfg["dir"] * idx * cfg["step"]
    x_base = aa_to_x(row["plot_pos"])

    if n == 1:
        x_label = x_base
    elif n == 2:
        x_label = x_base + (idx - 0.5) * 60
    else:
        x_label = x_base + (idx - (n - 1) / 2) * 50

    if mt == "missense":
      y_label = y_pt - 30 - (idx % 3) * 8

    else:
      y_label = y_pt + 35 + (idx % 3) * 10

    points.append({
        "x": x_base,
        "y": y_pt,
        "x_label": x_label,
        "y_label": y_label,
        **row.to_dict()
    })

max_y = max(p["y_label"] for p in points if p["mut_type"] != "missense") + 200
min_y = min(p["y_label"] for p in points if p["mut_type"] == "missense") - 60

SVG_H = int(max_y - min_y + 320)
OFFSET = -int(min_y) + 60

def ty(y):
    return y + OFFSET

BY = ty(BACKBONE_Y)
AXIS_Y = ty(max_y + 45)

#  6. Build SVG

svg = []

svg.append(
    f'<svg viewBox="0 0 {W} {SVG_H}" xmlns="http://www.w3.org/2000/svg" '
    f'style="width:100%;font-family:Arial,sans-serif;background:white">'
)

svg.append('''<defs>
  <pattern id="hatch" patternUnits="userSpaceOnUse" width="5" height="5" patternTransform="rotate(45)">
    <line x1="0" y1="0" x2="0" y2="5" stroke="rgba(255,255,255,0.5)" stroke-width="2"/>
  </pattern>
</defs>''')

# stems
for p in points:
    col = SZ_COLORS[p["seizure"]]
    svg.append(
        f'<line class="stem" data-sz="{col}" data-path="{PATH_COLORS[p["path_clean"]]}" '
        f'x1="{p["x"]:.1f}" y1="{BY}" x2="{p["x"]:.1f}" y2="{ty(p["y"]):.1f}" '
        f'stroke="{col}" stroke-width="0.9" opacity="0.6"/>'
    )

# connectors
for p in points:
    svg.append(
        f'<line class="conn" x1="{p["x"]:.1f}" y1="{ty(p["y"]):.1f}" '
        f'x2="{p["x_label"]:.1f}" y2="{ty(p["y_label"]):.1f}" '
        f'stroke="#ccc" stroke-width="0.5" opacity="0.8"/>'
    )

# grid
for aa in [500, 1000, 1500, 2000, 2500, 3000]:
    x = aa_to_x(aa)
    svg.append(
        f'<line x1="{x:.1f}" y1="{ty(min_y - 40)}" x2="{x:.1f}" y2="{AXIS_Y}" '
        f'stroke="#EEEEEE" stroke-width="1"/>'
    )

# backbone
svg.append(
    f'<line x1="{LEFT}" y1="{BY}" x2="{RIGHT}" y2="{BY}" '
    f'stroke="black" stroke-width="3"/>'
)

# domains
for d in domains:
    x1 = aa_to_x(d["start"])
    x2 = aa_to_x(d["end"])
    w = x2 - x1
    op = "1" if d["wd"] else "1"

    svg.append(
        f'<rect x="{x1:.1f}" y="{BY-21}" width="{w:.1f}" height="42" '
        f'fill="{d["color"]}" opacity="{op}" stroke="black" stroke-width="1"/>'
    )

    if d["wd"]:
        svg.append(
            f'<rect x="{x1:.1f}" y="{BY-21}" width="{w:.1f}" height="42" fill="url(#hatch)"/>'
        )

    if not d["wd"] and w > 30:
        mx = (x1 + x2) / 2
        svg.append(
            f'<text x="{mx:.1f}" y="{BY+6}" text-anchor="middle" '
            f'font-size="13" font-weight="bold" fill="white">{d["name"]}</text>'
        )


# N/C
svg.append(f'<text x="{LEFT-12}" y="{BY+5}" text-anchor="end" font-size="13" font-weight="bold">N</text>')
svg.append(f'<text x="{RIGHT+12}" y="{BY+5}" text-anchor="start" font-size="13" font-weight="bold">C</text>')

# row labels
for lbl, cfg in TYPE_Y.items():
    svg.append(
        f'<text x="{LEFT-10}" y="{ty(cfg["base"])+5}" text-anchor="end" '
        f'font-size="10" fill="#888" font-style="italic">{lbl.capitalize()}</text>'
    )

# points
for i, p in enumerate(points):
    mt = p["mut_type"]
    col = SZ_COLORS[p["seizure"]]
    pc = PATH_COLORS[p["path_clean"]]
    x, y = p["x"], ty(p["y"])
    r = 7

    svg.append(f'<g class="variant" data-i="{i}" data-sz="{col}" data-path="{pc}">')

    if mt == "missense":
        svg.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r}" fill="{col}" stroke="black" stroke-width="1"/>')
    elif mt == "nonsense":
        pts = f"{x:.1f},{y-r:.1f} {x-r*.87:.1f},{y+r*.5:.1f} {x+r*.87:.1f},{y+r*.5:.1f}"
        svg.append(f'<polygon points="{pts}" fill="{col}" stroke="black" stroke-width="1"/>')
    elif mt == "frameshift":
        svg.append(f'<rect x="{x-r:.1f}" y="{y-r:.1f}" width="{r*2}" height="{r*2}" fill="{col}" stroke="black" stroke-width="1"/>')
    elif mt == "splice":
        svg.append(f'<polygon points="{x:.1f},{y-r:.1f} {x+r:.1f},{y:.1f} {x:.1f},{y+r:.1f} {x-r:.1f},{y:.1f}" fill="{col}" stroke="black" stroke-width="1"/>')

    if p["inherited"]:
        svg.append(f'<circle class="inh" cx="{x:.1f}" cy="{y:.1f}" r="2.5" fill="black"/>')

    svg.append('</g>')

# labels
for p in points:
    col = SZ_COLORS[p["seizure"]]
    pc = PATH_COLORS[p["path_clean"]]
    lx = p["x_label"]
    ly = ty(p["y_label"])
    lbl = p["label"].replace("p.", "") if str(p["label"]).startswith("p.") else p["label"]

    svg.append(
        f'<text class="var-label" data-sz="{col}" data-path="{pc}" '
        f'x="{lx:.1f}" y="{ly:.1f}" text-anchor="end" font-size="8" '
        f'fill="{col}" font-weight="500" transform="rotate(-45,{lx:.1f},{ly:.1f})">{lbl}</text>'
    )


# 7. IGD bars

nbea_exon_aa = {str(i + 1): aa for i, aa in enumerate([
    1, 18, 46, 72, 102, 134, 168, 205,
    244, 284, 326, 370, 416, 464, 514, 566,
    620, 676, 734, 794, 856, 920, 986, 1054,
    1124, 1196, 1270, 1346, 1424, 1504, 1586, 1670,
    1756, 1844, 1934, 2026, 2120, 2216, 2314, 2414,
    2516, 2620, 2726, 2834, 2898, 2961
])}

igd = df2[df2["mut_type"].str.contains("igd", na=False)].copy()

igd["igd_start"] = pd.to_numeric(igd["igd_exon_start"], errors="coerce")
igd["igd_end"] = pd.to_numeric(igd["igd_exon_end"], errors="coerce")

igd = igd[igd["igd_start"].notna()].copy()
igd["igd_end"] = igd["igd_end"].fillna(igd["igd_start"])

igd = igd.sort_values(
    ["igd_start", "igd_end"],
    ascending=[True, False]
).reset_index(drop=True)

print("IGD rows being plotted:")
print(igd[["mut_type", "igd_exon_start", "igd_exon_end", "igd_start", "igd_end"]])

IGD_BASE_Y = max_y -60
IGD_GAP = 22

for idx2, row in igd.iterrows():
    start_key = str(int(row["igd_start"]))
    end_key = str(int(row["igd_end"]))

    if start_key not in nbea_exon_aa or end_key not in nbea_exon_aa:
        continue

    x1 = aa_to_x(nbea_exon_aa[start_key])
    x2 = aa_to_x(nbea_exon_aa[end_key])

    if x2 < x1:
        x1, x2 = x2, x1

    if abs(x2 - x1) < 8:
        x2 = x1 + 8

    y = ty(IGD_BASE_Y + idx2 * IGD_GAP)
    col = SZ_COLORS[row["seizure"]]
    lbl = f'Ex{int(row["igd_start"])}-{int(row["igd_end"])} del'

    svg.append(
        f'<line class="igd" data-sz="{col}" data-path="{PATH_COLORS[row["path_clean"]]}" '
        f'x1="{x1:.1f}" y1="{y:.1f}" x2="{x2:.1f}" y2="{y:.1f}" '
        f'stroke="{col}" stroke-width="3"/>'
    )

    svg.append(
        f'<text x="{x1-4:.1f}" y="{y+4:.1f}" text-anchor="end" '
        f'font-size="8" fill="{col}">{lbl}</text>'
    )
AXIS_Y = ty(IGD_BASE_Y + len(igd) * IGD_GAP + 25)

# axis
svg.append(
    f'<line x1="{LEFT}" y1="{AXIS_Y}" x2="{RIGHT}" y2="{AXIS_Y}" stroke="#ccc" stroke-width="1"/>'
)

for aa in [0, 500, 1000, 1500, 2000, 2500, 3000]:
    x = aa_to_x(aa)
    svg.append(f'<line x1="{x:.1f}" y1="{AXIS_Y}" x2="{x:.1f}" y2="{AXIS_Y+6}" stroke="#bbb" stroke-width="1"/>')
    svg.append(f'<text x="{x:.1f}" y="{AXIS_Y+18}" text-anchor="middle" font-size="9" fill="#aaa">{aa}</text>')

svg.append(
    f'<text x="{(LEFT+RIGHT)/2}" y="{AXIS_Y+32}" text-anchor="middle" '
    f'font-size="10" fill="#999" font-style="italic">Amino acid position</text>'
)


svg.append("</svg>")
svg_content = "\n".join(svg)

#  8. HTML

data_json = json.dumps(pt.to_dict("records"))
sz_json = json.dumps(SZ_COLORS)
path_json = json.dumps(PATH_COLORS)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>NBEA Variant Landscape</title>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'Helvetica Neue',Arial,sans-serif;background:#f4f4f0;padding:20px}}
h1{{font-size:22px;font-weight:700;color:#111;margin-bottom:4px}}
.subtitle{{font-size:11px;color:#888;font-style:italic;margin-bottom:16px}}
.chart-wrap{{background:#fff;border-radius:8px;box-shadow:0 2px 12px rgba(0,0,0,.1);padding:16px;overflow-x:auto}}
.toggle-wrap{{display:flex;gap:10px;margin-bottom:14px;align-items:center}}
.toggle-wrap span{{font-size:11px;color:#666;font-weight:600}}
button{{padding:6px 14px;border-radius:5px;border:1px solid #ccc;background:#f8f8f8;cursor:pointer;font-size:11px;font-weight:600}}
button.active{{background:#1a1a1a;color:white;border-color:#1a1a1a}}
.legend-wrap{{display:flex;flex-wrap:wrap;gap:10px 28px;margin-top:14px;background:#fff;border-radius:6px;box-shadow:0 1px 6px rgba(0,0,0,.08);padding:12px 16px}}
.lg{{display:flex;flex-direction:column;gap:4px}}
.lgh{{font-size:8px;font-weight:700;text-transform:uppercase;letter-spacing:.09em;color:#aaa;margin-bottom:2px}}
.lgi{{display:flex;align-items:center;gap:6px;font-size:10px;color:#333}}
.lsw{{width:12px;height:12px;flex-shrink:0;border-radius:2px}}
#tooltip{{position:fixed;pointer-events:none;background:rgba(10,10,20,.93);color:#e8e8e8;border-radius:8px;padding:11px 14px;font-size:11px;max-width:300px;z-index:9999;display:none;line-height:1.55;box-shadow:0 4px 20px rgba(0,0,0,.45)}}
#tooltip .tt-hdr{{font-size:13px;font-weight:700;color:#fff;margin-bottom:4px}}
#tooltip .tt-badge{{display:inline-block;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700;color:#fff;margin-bottom:6px}}
#tooltip .tt-row{{font-size:10px;color:#aaa;margin-bottom:2px}}
#tooltip .tt-row b{{color:#ddd}}
</style>
</head>
<body>

<h1>NBEA Variant Distribution Lollipop Plot</h1>
<p class="subtitle">Transcript: NM_015678.4 &nbsp;&middot;&nbsp; Protein: NP_056493.3 &nbsp;&middot;&nbsp; Hover over any variant for details</p>

<div class="toggle-wrap">
  <span>Color by:</span>
  <button id="btn-sz" class="active" onclick="setMode('sz')">Seizure Status</button>
  <button id="btn-path" onclick="setMode('path')">Pathogenicity</button>
</div>

<div class="chart-wrap">
{svg_content}
</div>

<div class="legend-wrap">
  <div class="lg">
    <div class="lgh">Seizure status</div>
    <div class="lgi"><div class="lsw" style="background:#D62728;border-radius:50%"></div>Seizure</div>
    <div class="lgi"><div class="lsw" style="background:#4878CF;border-radius:50%"></div>No seizure</div>
    <div class="lgi"><div class="lsw" style="background:#AAAAAA;border-radius:50%"></div>Unknown</div>
  </div>

  <div class="lg">
    <div class="lgh">Pathogenicity</div>
    <div class="lgi"><div class="lsw" style="background:#C0392B;border-radius:50%"></div>Pathogenic</div>
    <div class="lgi"><div class="lsw" style="background:#F39C12;border-radius:50%"></div>VUS</div>
    <div class="lgi"><div class="lsw" style="background:#AAAAAA;border-radius:50%"></div>Unknown</div>
  </div>

  <div class="lg">
    <div class="lgh">Variant type</div>
    <div class="lgi">● Missense</div>
    <div class="lgi">▲ Nonsense</div>
    <div class="lgi">■ Frameshift</div>
    <div class="lgi">◆ Splice</div>
  </div>


  <div class="lg">
    <div class="lgh">Domains</div>
    <div class="lgi"><div class="lsw" style="background:#4C72B0"></div>D4704</div>
    <div class="lgi"><div class="lsw" style="background:#F1C40F"></div>D1088</div>
    <div class="lgi"><div class="lsw" style="background:#8E44AD"></div>PH</div>
    <div class="lgi"><div class="lsw" style="background:#C0392B"></div>BEACH</div>
    <div class="lgi"><div class="lsw" style="background:#2E7D32"></div>WD repeats</div>
  </div>

<div class="lg">
  <div class="lgh">Notes</div>
  <div class="lgi" style="font-size:9px;color:#999;line-height:1.5">
    Black centre dot = inherited variant<br>
    Hover any variant for clinical details<br>
    Toggle buttons switch color encoding
  </div>
</div>

<div id="tooltip"></div>

<script>
const SZ_COLORS = {sz_json};
const PATH_COLORS = {path_json};
const DATA = {data_json};
let mode = 'sz';

function setMode(m) {{
  mode = m;
  document.getElementById('btn-sz').className = m === 'sz' ? 'active' : '';
  document.getElementById('btn-path').className = m === 'path' ? 'active' : '';

  document.querySelectorAll('.stem, .igd').forEach(el => {{
    el.setAttribute('stroke', m === 'sz' ? el.dataset.sz : el.dataset.path);
  }});

  document.querySelectorAll('.variant').forEach(el => {{
    const col = m === 'sz' ? el.dataset.sz : el.dataset.path;
    el.querySelectorAll('circle:not(.inh), polygon, rect').forEach(s => s.setAttribute('fill', col));
  }});

  document.querySelectorAll('.var-label').forEach(el => {{
    el.setAttribute('fill', m === 'sz' ? el.dataset.sz : el.dataset.path);
  }});
}}

const TT = document.getElementById('tooltip');

document.querySelectorAll('.variant').forEach(el => {{
  el.style.cursor = 'pointer';
  el.addEventListener('mouseenter', function(e) {{
    const i = parseInt(this.dataset.i);
    const d = DATA[i];
    const col = mode === 'sz' ? SZ_COLORS[d.seizure] : PATH_COLORS[d.path_clean];
    const mt = d.mut_type.charAt(0).toUpperCase() + d.mut_type.slice(1);

    TT.innerHTML = `
      <div class="tt-hdr">${{d.label || d.coding}}</div>
      <span class="tt-badge" style="background:${{col}}">${{mt}}</span>
      <div class="tt-row"><b>Seizure:</b> ${{d.seizure}}</div>
      <div class="tt-row"><b>Pathogenicity:</b> ${{d.path_clean}}</div>
      <div class="tt-row"><b>Inheritance:</b> ${{d.inherited ? 'Inherited' : 'De novo / Unknown'}}</div>
      ${{d.onset ? `<div class="tt-row"><b>Seizure onset:</b> ${{d.onset}}</div>` : ''}}
      ${{d.coding ? `<div class="tt-row"><b>Coding:</b> ${{d.coding}}</div>` : ''}}
    `;

    TT.style.display = 'block';
    moveTip(e);
  }});

  el.addEventListener('mouseleave', () => TT.style.display = 'none');
  el.addEventListener('mousemove', moveTip);
}});

function moveTip(e) {{
  const w = TT.offsetWidth || 280;
  const h = TT.offsetHeight || 160;
  const off = 14;

  let l = e.clientX + off;
  let t = e.clientY + off;

  if (l + w > window.innerWidth - 8) l = e.clientX - w - off;
  if (t + h > window.innerHeight - 8) t = e.clientY - h - off;

  TT.style.left = l + 'px';
  TT.style.top = t + 'px';
}}
</script>
</body>
</html>
"""

# 9. Save


out_path = "/content/drive/MyDrive/nbea_interactive3.html"

with open(out_path, "w") as f:
    f.write(html)

print("Saved:", out_path)
print(f"Total point variants plotted: {len(points)}")
print(f"Total IGD rows plotted: {len(igd)}")

IGD rows being plotted:
  mut_type  igd_exon_start  igd_exon_end  igd_start  igd_end
0      igd             2.0          46.0        2.0     46.0
1      igd             2.0          38.0        2.0     38.0
2      igd            18.0          36.0       18.0     36.0
3      igd            18.0          36.0       18.0     36.0
4      igd            28.0          32.0       28.0     32.0
5      igd            39.0          42.0       39.0     42.0
6      igd            41.0          45.0       41.0     45.0
Saved: /content/drive/MyDrive/nbea_interactive3.html
Total point variants plotted: 81
Total IGD rows plotted: 7


In [21]:
from google.colab import files
files.download("/content/drive/MyDrive/nbea_interactive3.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
print("Points plotted (variants):", len(points))
print("IGD bars:", len(igd))
print("Total:", len(points) + len(igd))

Points plotted (variants): 81
IGD bars: 7
Total: 88
